# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [9]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.



In [10]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [11]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel won the F1 2010 championship driving for Red Bull Racing. It was his first of four consecutive world championships.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [21]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

Driver: Sebastian Vettel.
Team: Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [24]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


In [26]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton.
Team: Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [12]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton.
Team: Mercedes.
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [32]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [33]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.
Team: Renault.


Few Shots for classification.


In [13]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))

Sentiment: Negative


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [ ]:
# Version 1: Customer Support Bot

context_support = [
    {
        'role': 'system', 
        'content': 'You are an AI customer support assistant. Your task is to analyze incoming messages, determine the category (Return, Delivery Status, Complaint), and extract the order number. Adhere strictly to the format.'
    },
    {
        'role': 'user', 
        'content': 'Hello, my order #10234 still has not arrived. Can you please check on it?'
    },
    {
        'role': 'assistant', 
        'content': 'Category: Delivery Status\nOrder Number: 10234'
    },
    {
        'role': 'user', 
        'content': 'The blue pants do not fit at all, so I am sending them back tomorrow. My invoice number was 99482.'
    },
    {
        'role': 'assistant', 
        'content': 'Category: Return\nOrder Number: 99482'
    }
]

# Test Case A: An angry customer without a provided order number
test_message_1 = "I am absolutely dissatisfied with the quality of this sweater! It shrunk completely after the first wash. This is unacceptable!"
print("--- Test 1 (Complaint without number) ---")
print(return_OAIResponse(test_message_1, context_support))

# Test Case B: A message containing a potentially confusing number (not an order number)
test_message_2 = "Hello, I have been waiting for my package for 3 days now. Where is it?"
print("\n--- Test 2 (Delivery status with distractor number) ---")
print(return_OAIResponse(test_message_2, context_support))

--- Test 1 (Complaint without number) ---
Category: Complaint

--- Test 2 (Delivery status with distractor number) ---
Category: Delivery Status


In [15]:
# Version 2: Recipe Converter (Structuring Unstructured Text)
context_recipe = [
    {
        'role': 'system', 
        'content': 'You are an AI that converts messy, conversational cooking descriptions into structured recipe data. Format your output exactly like this:\nRecipe Name: [Name]\nIngredients:\n- [Ingredient 1]\n- [Ingredient 2]\nInstructions:\n1. [Step 1]\n2. [Step 2]'
    },
    {
        'role': 'user', 
        'content': 'So yesterday I made this quick pasta. I just boiled like 200g of spaghetti, fried 3 cloves of minced garlic in olive oil, and threw it all together with some parmesan cheese.'
    },
    {
        'role': 'assistant', 
        'content': 'Recipe Name: Quick Garlic Pasta\nIngredients:\n- 200g spaghetti\n- 3 cloves garlic (minced)\n- Olive oil\n- Parmesan cheese\nInstructions:\n1. Boil 200g of spaghetti.\n2. Fry minced garlic in olive oil.\n3. Mix the pasta with the garlic oil and parmesan cheese.'
    }
]

# Test Case: A messy, chaotic text where ingredients are hidden and steps are out of order
test_recipe_text = "To make my grandma's apple pancakes, you actually need to slice two apples first. But wait, before that, mix 200g of flour with two eggs and a splash of milk in a bowl. Then throw the apples into the pan, pour the batter over them, and bake until golden."

print("--- Test: Recipe Converter ---")
print(return_OAIResponse(test_recipe_text, context_recipe))

--- Test: Recipe Converter ---
Recipe Name: Grandma's Apple Pancakes
Ingredients:
- 2 apples
- 200g flour
- 2 eggs
- Milk
Instructions:
1. Slice two apples.
2. In a bowl, mix 200g of flour with two eggs and a splash of milk.
3. Place the sliced apples in a pan.
4. Pour the batter over the apples.
5. Bake until golden brown.


In [16]:
# Version 3: Medical Report Simplification (Latin/Greek to Patient Layman Terms)
context_medical = [
    {
        'role': 'system', 
        'content': 'You are an empathetic physician translating complex medical reports written in Latin/Greek terminology into clear, easily understandable language for patients. Maintain a reassuring yet accurate tone. Format the output with: "Diagnosis:" and "Explanation:".'
    },
    {
        'role': 'user', 
        'content': 'Patient presents with acute appendicitis with signs of beginning perforation.'
    },
    {
        'role': 'assistant', 
        'content': 'Diagnosis: Acute appendicitis (inflammation of the appendix)\nExplanation: Your appendix is severely inflamed. There are early signs that the wall of the organ is weakening and could tear. This requires prompt medical attention.'
    },
    {
        'role': 'user', 
        'content': 'Radiological findings indicate severe bilateral coxarthrosis with joint space narrowing and osteophyte formation.'
    },
    {
        'role': 'assistant', 
        'content': 'Diagnosis: Chronic wear and tear of both hip joints\nExplanation: The X-rays show significant cartilage loss in both of your hips. The protective cushion in the joint has thinned, and in response, the bone has developed small, bony growths.'
    }
]

# Test Case: A highly complex neurological/orthopedic description with deep Latin/Greek roots
test_medical_text = "The MRI of the lumbar spine reveals a mediolateral disc prolaps at L4/L5 with consecutive compression of the L5 radix and pronounced spinal canal stenosis."

print("--- Test: Medical Simplification ---")
print(return_OAIResponse(test_medical_text, context_medical))

--- Test: Medical Simplification ---
Diagnosis: Disc herniation at L4/L5 level causing pressure on the nerve and narrowing of the spinal canal
Explanation: The MRI shows that a disc between two of your lower back bones has slipped out of place to the side. It is pressing on a nerve and causing narrowing of the canal that holds the spinal cord.


### First Version: Customer Support Classification (Few-Shot Prompting)

*   **Approach:** I tested few-shot prompting to classify customer support messages and extract order numbers.
*   **Result:** The model correctly identified the categories (e.g., Complaint, Delivery Status).
*   **The Issue:** I did not include a negative example (a message missing an order number) within the few-shot context.
*   **Observation:** Instead of hallucinating a fake order number as expected, the model completely omitted the "Order Number" line from its response, failing to maintain the requested output format.

### Second Version: Recipe Converter (Structuring Unstructured Text)

*   **Approach:** I tested the model's ability to extract information from a chaotic, non-chronological cooking description and format it into a structured recipe (Name, Ingredients, Instructions).
*   **Result:** The model successfully maintained the requested output format and correctly identified all ingredients.
*   **The Issue (Logical Failure):** The test text contained a chronological trap (*"slice apples first... but wait, before that, mix the batter"*). The model failed to process the *"before that"* modifier.
*   **Observation:** It listlessly copied the steps in the order they appeared in the text, resulting in an incorrect logical sequence where the user cuts apples before mixing the dough that goes under them. It also dropped the measurement modifier for the milk (listing just "Milk" instead of "A splash of milk").

### Third Version: Medical Report Simplification (Latin/Greek to Layman Terms)

*   **Approach:** I tested few-shot prompting to evaluate the model's capacity to translate highly technical medical terminology (with deep Latin/Greek roots) into clear, reassuring, and easily understandable language for patients while retaining structural formatting.
*   **Result:** The model performed exceptionally well. It adhered perfectly to the requested "Diagnosis:" and "Explanation:" syntax.
*   **Observation:** Unlike the previous tests, this variation worked flawlessly. The model correctly mapped *disc prolaps* to "slipped disc," *radix* to "nerve," and *spinal canal stenosis* to "narrowing of the canal." It successfully maintained the specific anatomical location (L4/L5) without dropping crucial data or hallucinating extraneous medical conditions. The tone was empathetic yet scientifically accurate.

### Overall Key Takeaways on Prompting Strategies

Through these experiments, I learned that while Zero-Shot prompting relies entirely on the model’s internal knowledge base, introducing One-Shot and Few-Shot examples is critical for enforcing structural and stylistic guardrails in inference[cite: 1]. However, Few-Shot prompting is highly sensitive to the diversity of the context data provided. Positive examples alone are insufficient; if the model is not explicitly shown a negative example (such as an edge case where requested data is missing), it may break formatting rules or omit fields entirely rather than gracefully handling the exception. 

Furthermore, while Few-Shot learning is highly effective at teaching a model style, tone, and vocabulary mapping—as demonstrated by the flawless medical translation—it cannot fully compensate for deep reasoning flaws when a prompt contains complex, non-linear logic or temporal contradictions. Ultimately, successful prompt engineering requires balancing explicit instructional boundaries with a comprehensive set of examples that anticipate both data gaps and logical constraints.